In [3]:
import datar.all as dr
from datar import f

import polars as pl

import numpy as np
from scipy import stats

from pathlib import Path
import warnings

from pipda import register_verb

dr.filter = register_verb(func=dr.filter_)

warnings.filterwarnings("ignore")
pl.Config.set_tbl_cell_alignment("CENTER")

data_dir = next(Path("/home/").glob("**/datar-polars/data"))

In [4]:
df_baseball = (
    dr.tibble(
        pl.read_csv(
            source=data_dir/"baseball.csv",
            columns=["Name", "Team", "Height", "Weight"],
        )
    )
    >> dr.mutate(Team = dr.as_factor(f.Team))
)

print(df_baseball >> dr.slice_head(4))

shape: (4, 4)
┌─────────────────┬──────┬────────┬────────┐
│       Name      ┆ Team ┆ Height ┆ Weight │
│       ---       ┆  --- ┆   ---  ┆   ---  │
│       str       ┆  cat ┆   i64  ┆   i64  │
╞═════════════════╪══════╪════════╪════════╡
│  Adam_Donachie  ┆  BAL ┆   74   ┆   180  │
│    Paul_Bako    ┆  BAL ┆   74   ┆   215  │
│ Ramon_Hernandez ┆  BAL ┆   72   ┆   210  │
│   Kevin_Millar  ┆  BAL ┆   72   ┆   210  │
└─────────────────┴──────┴────────┴────────┘


# <span style="color:#1E90FF">1. dr.pipe(): apply custom functions directly</span>

In [7]:
print(
    df_baseball
    >> dr.filter((f.Height > 75) & (f.Weight <= 200))
    >> dr.pipe(
        lambda f: f.rename(lambda col: col.upper())
    )  # rename the columns to uppercase (use polars method)
    >> dr.slice_head(4)
)

shape: (4, 4)
┌────────────────┬──────┬────────┬────────┐
│      NAME      ┆ TEAM ┆ HEIGHT ┆ WEIGHT │
│       ---      ┆  --- ┆   ---  ┆   ---  │
│       str      ┆  cat ┆   i64  ┆   i64  │
╞════════════════╪══════╪════════╪════════╡
│   Kris_Benson  ┆  BAL ┆   76   ┆   195  │
│   James_Hoey   ┆  BAL ┆   78   ┆   200  │
│  Ryan_Sweeney  ┆  CWS ┆   76   ┆   200  │
│ Mike_MacDougal ┆  CWS ┆   76   ┆   195  │
└────────────────┴──────┴────────┴────────┘


# <span style="color:#1E90FF">2. dr.pipe(): apply user-defined functions</span>

In [9]:
def bmi_calculate(df):
    bmi = df.Weight / (df.Height**2) * 703
    return df.with_columns(BMI=bmi)

print(
    df_baseball
    >> dr.pipe(bmi_calculate)  # apply user-defined function directly in the pipe chain
    >> dr.slice_head(4)
)

shape: (4, 5)
┌─────────────────┬──────┬────────┬────────┬───────────┐
│       Name      ┆ Team ┆ Height ┆ Weight ┆    BMI    │
│       ---       ┆  --- ┆   ---  ┆   ---  ┆    ---    │
│       str       ┆  cat ┆   i64  ┆   i64  ┆    f64    │
╞═════════════════╪══════╪════════╪════════╪═══════════╡
│  Adam_Donachie  ┆  BAL ┆   74   ┆   180  ┆ 23.108108 │
│    Paul_Bako    ┆  BAL ┆   74   ┆   215  ┆ 27.601351 │
│ Ramon_Hernandez ┆  BAL ┆   72   ┆   210  ┆ 28.478009 │
│   Kevin_Millar  ┆  BAL ┆   72   ┆   210  ┆ 28.478009 │
└─────────────────┴──────┴────────┴────────┴───────────┘


# <span style="color:#1E90FF">3. dr.pipe(): more examples with ``scipy.stats`` functions</span>

The cumulative distribution function (CDF) takes a value and returns the probability
that a random variable is less than or equal to that value;

The percent-point function (PPF), also called the inverse CDF or quantile function,
takes a probability and returns the corresponding value whose CDF equals that probability.

In short: CDF input is a value and output is a probability in;
PPF input is a probability in and output is a value on the distribution's scale.

########################

In this example,  for the sake of reframing demonstration,
we will calculate the PPF values for the 25th, 50th, 75th, and 100th percentiles,
but for different distributions: normal and gamma.

height ~ normal distribution
weight ~ gamma distribution

In [10]:

print(
    df_baseball
    >> dr.pipe(lambda f: f >> dr.reframe(
       # Use ``dr.pipe()`` with ``lambda f:`` to get eager direct access and bypass the polars expression, 
       # so that ``stats`` functions can work
            height_norm=stats.norm.ppf(q=[0.25, 0.5, 0.75, 1], loc=f["Height"].mean(), scale=f["Height"].std()),
            weight_gamma=stats.gamma.ppf(q=[0.25, 0.5, 0.75, 1], a=2, scale=f["Weight"].mean() / 2)
    ))
    >> dr.add_column(index=["ppf_25th", "ppf_50th", "ppf_75th", "ppf_100th"], _before=0)
)

shape: (4, 3)
┌───────────┬─────────────┬──────────────┐
│   index   ┆ height_norm ┆ weight_gamma │
│    ---    ┆     ---     ┆      ---     │
│    str    ┆     f64     ┆      f64     │
╞═══════════╪═════════════╪══════════════╡
│  ppf_25th ┆  72.128932  ┆   96.776148  │
│  ppf_50th ┆  73.689655  ┆   168.96655  │
│  ppf_75th ┆  75.250379  ┆  271.079323  │
│ ppf_100th ┆     inf     ┆      inf     │
└───────────┴─────────────┴──────────────┘
